In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from fxml import config
from fxml.datasets.splits import walk_forward_splits
from fxml.datasets.table import build_table, build_vol_table
from fxml.db.connection import connect
from fxml.db.repository import load_bars
from fxml.evaluation import plots
from fxml.evaluation.backtest import backtest
from fxml.features.volatility import VOL_FEATURE_COLUMNS
from fxml.training.baselines import walk_forward_predict
from fxml.training.experiment import Settings, run_walk_forward

plots.apply_style()
conn = connect()
dev_bars = load_bars(conn, "EURUSD", "H1", start="2001-01-01", end="2015-01-01")
print(len(dev_bars), "bars:", dev_bars.index.min(), "->", dev_bars.index.max())


86847 bars: 2001-01-01 06:00:00+00:00 -> 2014-12-31 17:00:00+00:00


In [2]:
HORIZON = 12
settings = Settings(horizon=HORIZON)
table = build_table(dev_bars, horizon=HORIZON)
folds = walk_forward_splits(len(table), train_size=31_000, test_size=3_100, gap=HORIZON)
wf = run_walk_forward(table, folds, settings, verbose=False)
preds = wf.predictions
print(len(preds), "out-of-sample direction predictions")


52700 out-of-sample direction predictions


In [3]:
vol_table = build_vol_table(dev_bars, horizon=config.VOL_HORIZON)
vol_folds = walk_forward_splits(len(vol_table), train_size=31_000, test_size=3_100,
                                gap=config.VOL_HORIZON)
vol_forecast = walk_forward_predict(vol_table, vol_folds, VOL_FEATURE_COLUMNS)
print(len(vol_forecast), "out-of-sample volatility forecasts")


52700 out-of-sample volatility forecasts


In [4]:
joined = preds.join(vol_forecast, how="inner")
print(len(preds), "direction,", len(vol_forecast), "volatility,", len(joined), "in both")

FLAT_COST = 0.8e-4  # 0.8 bps per trade
cost = pd.Series(FLAT_COST, index=joined.index)
confidence = (joined["prob_up"] - 0.5).abs()
conf_threshold = confidence.quantile(0.8)
vol_threshold = joined["forecast"].median()


def run(mask, label):
    p = joined[["prob_up", "actual_return"]].copy()
    p.loc[~mask, "prob_up"] = 0.5  # no signal -> flat
    s = backtest(p, cost, conf_threshold, HORIZON).stats
    slipped = backtest(p, cost, conf_threshold, HORIZON, extra_cost_bps=0.3).stats
    keep = ["n_trades", "hit_rate", "gross_bps_per_trade", "net_bps_per_trade",
            "sharpe", "max_drawdown"]
    return {"scenario": label, **{k: s[k] for k in keep},
            "net_bps_with_slippage": slipped["net_bps_per_trade"]}


results = pd.DataFrame([
    run(pd.Series(True, index=joined.index), "A: confidence only"),
    run(joined["forecast"] > vol_threshold, "B: + high forecast vol"),
    run(joined["forecast"] <= vol_threshold, "C: + low forecast vol"),
]).set_index("scenario")
results


52700 direction, 52700 volatility, 52212 in both


,n_trades,hit_rate,gross_bps_per_trade,net_bps_per_trade,sharpe,max_drawdown,net_bps_with_slippage
scenario,,,,,,,
A: confidence only,10443,0.539500,1.958656,1.158656,0.832887,-0.112147,0.858656
B: + high forecast vol,4836,0.546112,2.264001,1.464001,0.552890,-0.112147,1.164001
C: + low forecast vol,5607,0.533797,1.695298,0.895298,0.731557,-0.047668,0.595298


In [5]:
gain = results.loc["B: + high forecast vol", "sharpe"] - results.loc["A: confidence only", "sharpe"]
print(f"B minus A net Sharpe: {gain:+.2f}   (pass needs >= +0.20)")


B minus A net Sharpe: -0.28   (pass needs >= +0.20)
